# Model Evaluation Notebook

This notebook evaluates the poultry detection models:
- Detection model (YOLOv12n)
- Pose estimation model (YOLO11-pose)
- Segmentation model (YOLO-seg)

**Metrics computed:**
- mAP50, mAP50-95
- Precision, Recall per class
- Confusion matrix
- Speed benchmarks (FPS)

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import time
import cv2

from ultralytics import YOLO

## 1. Load Models

In [ ]:
# Detection model
detection_model_path = Path('../models/poultry-yolov12n-v1.pt')
detection_model = YOLO(str(detection_model_path)) if detection_model_path.exists() else None
print(f"Detection model loaded: {detection_model is not None}")

# Pose model
pose_model_path = Path('../models/yolo11-pose.pt')
pose_model = YOLO(str(pose_model_path)) if pose_model_path.exists() else None
print(f"Pose model loaded: {pose_model is not None}")

# Segmentation model
seg_model_path = Path('../models/poultry-seg-large.pt')
seg_model = YOLO(str(seg_model_path)) if seg_model_path.exists() else None
print(f"Segmentation model loaded: {seg_model is not None}")

## 2. Model Information

In [ ]:
if detection_model:
    print("Detection Model Classes:")
    for idx, name in detection_model.names.items():
        print(f"  {idx}: {name}")
    
    # Model architecture info
    print(f"\nModel file size: {detection_model_path.stat().st_size / 1e6:.2f} MB")

## 3. Speed Benchmarks

In [ ]:
def benchmark_model(model, n_runs=50, img_size=640):
    """Benchmark inference speed."""
    if model is None:
        return None
    
    # Create dummy image
    dummy_img = np.random.randint(0, 255, (img_size, img_size, 3), dtype=np.uint8)
    
    # Warmup
    for _ in range(5):
        model(dummy_img, verbose=False)
    
    # Benchmark
    times = []
    for _ in range(n_runs):
        start = time.time()
        model(dummy_img, verbose=False)
        times.append(time.time() - start)
    
    avg_time = np.mean(times) * 1000  # ms
    fps = 1000 / avg_time
    
    return {
        'avg_time_ms': avg_time,
        'fps': fps,
        'std_ms': np.std(times) * 1000
    }

print("Benchmarking detection model...")
det_bench = benchmark_model(detection_model)
if det_bench:
    print(f"  Avg inference time: {det_bench['avg_time_ms']:.2f} ms")
    print(f"  FPS: {det_bench['fps']:.1f}")

## 4. Sample Inference

In [ ]:
# Load a sample image
sample_images = list(Path('../sampleimages').glob('*.jpg')) + list(Path('../sampleimages').glob('*.png'))
sample_images += list(Path('../poultry-data/images').glob('*.png'))[:5]

if sample_images and detection_model:
    img_path = sample_images[0]
    print(f"Running inference on: {img_path}")
    
    results = detection_model(str(img_path))
    
    # Display
    annotated = results[0].plot()
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title('Detection Results')
    plt.axis('off')
    plt.show()
    
    # Print detections
    print("\nDetections:")
    for box in results[0].boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        print(f"  {detection_model.names[cls]}: {conf:.2f}")
else:
    print("No sample images found or model not loaded")

## 5. Validation (if dataset available)

In [ ]:
# Check if validation data exists
data_yaml = Path('../poultry-data/data.yaml')

if data_yaml.exists() and detection_model:
    print("Running validation...")
    metrics = detection_model.val(data=str(data_yaml))
    
    print(f"\nValidation Metrics:")
    print(f"  mAP50: {metrics.box.map50:.4f}")
    print(f"  mAP50-95: {metrics.box.map:.4f}")
else:
    print("Validation data not found. Export from pose-labeler first.")

## 6. Summary

In [ ]:
print("=" * 50)
print("Model Evaluation Summary")
print("=" * 50)

models_info = [
    ('Detection (YOLOv12n)', detection_model, detection_model_path),
    ('Pose (YOLO11-pose)', pose_model, pose_model_path),
    ('Segmentation', seg_model, seg_model_path)
]

for name, model, path in models_info:
    status = '✓' if model else '✗'
    size = f"{path.stat().st_size / 1e6:.1f} MB" if path.exists() else "N/A"
    print(f"{status} {name}: {size}")